# RAG with LangChain

This notebook was created when I was studying the LangChain library based in the book **Generative AI with LangChain, Second Edition** of the authors *Bean Aufffart* and *Leonid Kuligin*. The book is based on LangChain version 0.37.

## Preparing the API to Google's Gemini, OpenAI and others

I only have an API key for Google's Gemini, so every example will run on that model or in local model. The file that configures needs to contain the following code:

```python
import os

GOOGLE_API_KEY = "..."

def set_environment():
    variable_dict = globals().items()

    for key, value in variable_dict:
        if "API" in key or "ID" in key or "GOOGLE" in key:
            os.environ[key] = value
```

I will not upload this file to not risk of leak my own API key. The file `config.py` is listed on `.gitignore`.

So, all you need to do is to run the following code to enable all the API keys required to this notebook.

In [1]:
# setting the environment variables, the keys
import sys
import os

from IPython.core.debugger import prompt
from langchain_core.messages import HumanMessage
from openai import project
from watchfiles import awatch

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
# for the keys - as explained early in chapter 2
set_environment()

## Embeddings

Embeddings are numerical representation of text that capture semantic meaning. What makes embeddings powerful is that texts with similar meanings have similar numerical representation, enabling semantic search.

Here is an example on how to transform text in embeddings.

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize the embeddings model
embeddings_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Sentences from embedding
text1 = "The cat sat on the mat"
text2 = "A feline rested on the carpet"
text3 = "Python is a programming language"

# Get the embeddings
embeddings = embeddings_model.embed_documents([ text1, text2, text3 ])

embedding1 = embeddings[0]
embedding2 = embeddings[1]
embedding3 = embeddings[2]

print(f"Number of documents: {len(embeddings)}")
print(f"Dimensions per embedding: {len(embeddings[0])}")

/opt/homebrew/Caskroom/miniconda/base/envs/langchain_ai/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Number of documents: 3
Dimensions per embedding: 3072


## RAG

This is a simple RAG implementation:

In [6]:
from langchain.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# Basic RAG implementation
from langchain_community.document_loaders import JSONLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load documents
loader = JSONLoader(
    file_path="knowledge_base.json",
    jq_schema=".[].content",
    text_content=True
)
documents = loader.load()

# 2. Convert to vectors
#embedder = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
embedder = OpenAIEmbeddings()
embeddings = embedder.embed_documents([doc.page_content for doc in documents])

# 3. Store in vector database
vector_db = FAISS.from_documents(documents, embedder)

# 4. Retrieve similar docs
query   = "What are the effects of climate change?"
results = vector_db.similarity_search(query)

print(results)

[Document(id='c5b52fbd-b3e2-4ba6-9fcf-4ca102db66aa', metadata={'source': '/Users/asouza/repo/personal/ai-examples/lang-chain-python/knowledge_base.json', 'seq_num': 1}, page_content="Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows for more parallelization during training and better handling of long-range dependencies in text."), Document(id='66118883-b054-4356-8cd4-d73ff185ac10', metadata={'source': '/Users/asouza/repo/personal/ai-examples/lang-chain-python/knowledge_base.json', 'seq_num': 4}, page_content='Retrieval-Augmented Generation (RAG) combines a retrieval system with a text generator. The retriever fetches relevant documents from a knowledge base, and these documents are then provided as context to the generator. RAG models can be fine-tuned end-to-end and leverage large pre-trained models like BA

## Query transformation

Improving the queries

In [3]:
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

expansion_template = """
    Given the user question:  {question}
    Generate three alternative versions that express the same information need but with different wording:
    1.
"""

expansion_prompt = PromptTemplate(input_variables=["question"], template=expansion_template)

llm = ChatOpenAI(temperature=0.7)

expansion_chain = expansion_prompt | llm | StrOutputParser()

original_query = "What are the effects of climate change?"
expanded_queries = expansion_chain.invoke(original_query)

print(expanded_queries)

What impacts does climate change have on the environment?
2. How does climate change affect the planet?
3. What are the consequences of climate change on the Earth's ecosystems?


Hypothetical Document Embeddings (HyDE)

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="knowledge_base.json",
    jq_schema=".[].content",  # This extracts the content field from each array item
    text_content=True
)
documents = loader.load()
embedder = OpenAIEmbeddings()
embeddings = embedder.embed_documents([doc.page_content for doc in documents])
vector_db = FAISS.from_documents(documents, embedder)

In [5]:
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

hyde_template = """
    Based on the question:  {question}
    Write a passage that could contain the answer to this question:
    """

hyde_prompt = PromptTemplate(input_variables=["question"], template=hyde_template)

llm = ChatOpenAI(temperature=0.2)
hyde_chain = hyde_prompt | llm | StrOutputParser()

query = "What dietary changes can reduce carbon footprint?"
hypothetical_doc = hyde_chain.invoke(query)

embeddings = OpenAIEmbeddings()
embedded_query = embeddings.embed_query(hypothetical_doc)

results = vector_db.similarity_search_by_vector(embedded_query, k=3)
print(results)

[Document(id='f73af0a6-8f65-4409-a0bc-e5e2c31cfb65', metadata={'source': '/Users/asouza/repo/personal/ai-examples/lang-chain-python/knowledge_base.json', 'seq_num': 4}, page_content='Retrieval-Augmented Generation (RAG) combines a retrieval system with a text generator. The retriever fetches relevant documents from a knowledge base, and these documents are then provided as context to the generator. RAG models can be fine-tuned end-to-end and leverage large pre-trained models like BART or T5 for generation. This approach helps ground the generated text in factual information.'), Document(id='5a6eef50-3250-4b4a-82e2-c8f71fb3fab0', metadata={'source': '/Users/asouza/repo/personal/ai-examples/lang-chain-python/knowledge_base.json', 'seq_num': 1}, page_content="Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows fo

Contextual compression

In [8]:
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers import ContextualCompressionRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0)
compressor = LLMChainExtractor.from_llm(llm)

# Create a basis retriever from the vector store.
base_retriever = vector_db.as_retriever(search_Kwargs={"k": 3})

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

compressed_docs = compression_retriever.invoke("How do transformers work?")
print(compressed_docs)

[Document(metadata={'source': '/Users/asouza/repo/personal/ai-examples/lang-chain-python/knowledge_base.json', 'seq_num': 1}, page_content="Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows for more parallelization during training and better handling of long-range dependencies in text."), Document(metadata={'source': '/Users/asouza/repo/personal/ai-examples/lang-chain-python/knowledge_base.json', 'seq_num': 2}, page_content='BERT (Bidirectional Encoder Representations from Transformers) was developed by Google AI Language team in 2018.')]


Response enhancement

In [9]:
from langchain_core.documents import Document

# Some example documents
documents = [
    Document(
        page_content="The transform architecture was introduced in the paper 'Attention is All You Need' by Vaswani et al. in 2017.'",
        metadata={"source": "Neural Network Review 2021", "page": 42}
    ),
    Document(
        page_content="BERT uses bidirectional training of the Transformer, masked language modeling, and next sentence prediction tasks.",
        metadata={"source": "Introduction to NLP", "page": 137}
    ),
    Document(
        page_content="GPT models are autoregressive transformers that predict the next token based on previous tokens.",
        metadata={"source": "Large Language Models Survey", "page": 89}
    )
]

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# Create a vector store and retriever
embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(search_Kwargs={"k": 3})

# Source attribution prompt template
attribute_prompt = ChatPromptTemplate.from_template("""
    You are a precise AI assistant that provides well-sourced information.

    Answer the following question based ONLY on the provided sources. For each fact or claim in your answer, include a citation using
    [1], [2], etc. that refers to the source. Include a numbered list at the end.

    Question: {question}

    Sources:
    {sources}

    Your answer:
    """)

# Create a source-formatted string from documents
def format_sources_with_citations(docs):
    formatted_sources = []

    for i, doc in enumerate(docs, 1):
        source_info = f"[{i}] {doc.metadata.get('source', 'Unknown source')}"

        if doc.metadata.get('page'):
            source_info += f", page {doc.metadata[ 'page' ]}"
        formatted_sources.append(f"{source_info}\n{doc.page_content}")

    return "\n\n".join(formatted_sources)

# Build the RAG chain with source attribution
def generate_attributed_response(question):
    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # Format sources with citation numbers
    sources_formatted = format_sources_with_citations(retrieved_docs)

    # Create the attribution chain using LCEL
    attribution_chain = (
        attribute_prompt |
        ChatOpenAI(temperature=0) |
        StrOutputParser()
    )

    # Generate the response with citations
    response = attribution_chain.invoke({
        "question" : question,
        "sources"  : sources_formatted
    })

    return response


In [12]:
# Example usage
question = "How do transformer models work and what are some examples?"

attributed_answer = generate_attributed_response(question)
print(attributed_answer)

Transformer models work by utilizing a self-attention mechanism that allows them to weigh the importance of different input tokens when generating an output. This mechanism enables the model to capture long-range dependencies in the data more effectively compared to traditional recurrent neural networks [3].

Some examples of transformer models include BERT and GPT. BERT employs bidirectional training of the Transformer, masked language modeling, and next sentence prediction tasks [1]. On the other hand, GPT models are autoregressive transformers that predict the next token based on previous tokens [2].

1. The transformer architecture was introduced in the paper 'Attention is All You Need' by Vaswani et al. in 2017 [3].
2. BERT uses bidirectional training of the Transformer, masked language modeling, and next sentence prediction tasks [1].
3. GPT models are autoregressive transformers that predict the next token based on previous tokens [2].


This code not only ask a question to LLM but ask to verify the answer using the documents in the vector store

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from typing import List, Dict
from langchain_core.documents import Document

def verify_response_accuracy(
    retrieved_docs: List[Document],
    generated_answer: str,
    llm: ChatOpenAI = None
) -> Dict:
    """
    Verify if a generated answer is fully supported by the retrieved documents.
    Args:
        retrieved_docs: List of documents used to generate the answer
        generated_answer: The answer produced by the RAG system
        llm: Language model to use for verification
    Returns:
        Dictionary containing verification results and any identified issues
    """
    if llm is None:
        llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

    # Create context from retrieved documents
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # Define verification prompt - fixed to avoid JSON formatting issues in the template
    verification_prompt = ChatPromptTemplate.from_template("""
    As a fact-checking assistant, verify whether the following answer is fully supported
    by the provided context. Identify any statements that are not supported or contradict the context.

    Context:
    {context}

    Answer to verify:
    {answer}

    Perform a detailed analysis with the following structure:
    1. List any factual claims in the answer
    2. For each claim, indicate whether it is:
       - Fully supported (provide the supporting text from context)
       - Partially supported (explain what parts lack support)
       - Contradicted (identify the contradiction)
       - Not mentioned in context
    3. Overall assessment: Is the answer fully grounded in the context?

    Return your analysis in JSON format with the following structure:
    {{
      "claims": [
        {{
          "claim": "The factual claim",
          "status": "fully_supported|partially_supported|contradicted|not_mentioned",
          "evidence": "Supporting or contradicting text from context",
          "explanation": "Your explanation"
        }}
      ],
      "fully_grounded": true|false,
      "issues_identified": ["List any specific issues"]
    }}
    """)

    # Create verification chain using LCEL
    verification_chain = (
        verification_prompt
        | llm
        | StrOutputParser()
    )

    # Run verification
    result = verification_chain.invoke({
        "context": context,
        "answer": generated_answer
    })

    return result

# Example usage
retrieved_docs = [
    Document(page_content="The transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. It relies on self-attention mechanisms instead of recurrent or convolutional neural networks."),
    Document(page_content="BERT is a transformer-based model developed by Google that uses masked language modeling and next sentence prediction as pre-training objectives.")
]

generated_answer = "The transformer architecture was introduced by OpenAI in 2018 and uses recurrent neural networks. BERT is a transformer model developed by Google."

verification_result = verify_response_accuracy(retrieved_docs, generated_answer)
print(verification_result)

{
    "claims": [
        {
            "claim": "The transformer architecture was introduced by OpenAI in 2018",
            "status": "contradicted",
            "evidence": "The transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017.",
            "explanation": "The claim is contradicted by the fact that the transformer architecture was actually introduced in 2017 by Vaswani et al., not by OpenAI in 2018."
        },
        {
            "claim": "The transformer architecture uses recurrent neural networks",
            "status": "contradicted",
            "evidence": "It relies on self-attention mechanisms instead of recurrent or convolutional neural networks.",
            "explanation": "The claim is contradicted by the fact that the transformer architecture does not use recurrent neural networks, but rather self-attention mechanisms."
        },
        {
            "claim": "BERT is a transformer model developed by Google"

Testing the rag scripts created: